In [ ]:
!pip install fastapi uvicorn pyngrok google-genai

In [ ]:
import json
import os
import subprocess
import threading
import urllib.request
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from google import genai
from pydantic import BaseModel
import uvicorn

# FastAPIアプリの初期化
app = FastAPI()

# ScratchやWebサイト等、どこからでもアクセスできるようにCORS許可
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 認証キー設定
GEMINI_API_KEY = "Gemini API"
CLOUDFLARE_API_TOKEN = "Cloudflare API"
CLOUDFLARE_ACCOUNT_ID = "Cloudflare Account ID"

# Geminiクライアント準備
gemini_client = genai.Client(api_key=GEMINI_API_KEY)


# 受信データの形式（質問文を受け取る）
class QueryRequest(BaseModel):
    message: str


# Cloudflare Llama 3.1 呼び出し関数
def call_cloudflare(prompt: str) -> str:
    url = f"https://api.cloudflare.com/client/v4/accounts/{CLOUDFLARE_ACCOUNT_ID}/ai/run/@cf/meta/llama-3.1-8b-instruct-fast"
    payload = json.dumps(
        {"messages": [{"role": "user", "content": prompt}]}
    ).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=payload,
        headers={
            "Authorization": f"Bearer {CLOUDFLARE_API_TOKEN}",
            "Content-Type": "application/json",
        },
    )
    with urllib.request.urlopen(req) as res:
        data = json.loads(res.read().decode("utf-8"))
        return data["result"]["response"]


# Gemini 3.6 Flash 呼び出し関数
def call_gemini(prompt: str) -> str:
    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash", contents=prompt
    )
    return response.text


# 質疑応答エンドポイント
@app.post("/api/chat")
def qa_endpoint(req: QueryRequest):
    try:
        user_question = req.message

        # 1. 司令塔 (Cloudflare Llama 3.1) が質問を分析し、指示を作成
        commander_prompt = (
            f"あなたはAI Fusionの司令塔です。\n"
            f"以下の質問・依頼に対して、適切な回答を作成するための作業指示を箇条書きで出力してください。\n\n"
            f"質問：{user_question}"
        )
        commander_plan = call_cloudflare(commander_prompt)

        # 2. 実働 (Gemini 3.6 Flash) が指示に従って回答を作成（失敗時はCloudflareで自動補完）
        try:
            gemini_result = call_gemini(
                f"質問: {user_question}\n指示: {commander_plan}\n上記に従ってわかりやすく回答してください。"
            )
        except Exception:
            gemini_result = call_cloudflare(
                f"質問: {user_question}\n指示: {commander_plan}\n上記に従って回答を作成してください。"
            )

        # 3. 最終統合 (Cloudflare Llama 3.1) が回答を最終チェックして整形
        final_prompt = (
            f"ユーザーからの質問：{user_question}\n"
            f"生成された回答案：{gemini_result}\n\n"
            f"上記の内容を確認し、質問に対する分かりやすく正確な最終回答文を作成してください。"
        )
        final_answer = call_cloudflare(final_prompt)

        return {"status": "success", "response": final_answer}

    except Exception as e:
        return {"status": "error", "message": str(e)}


# バックグラウンドでAPIサーバーを起動
def run_app():
    uvicorn.run(app, host="127.0.0.1", port=8000)


threading.Thread(target=run_app, daemon=True).start()

# localtunnelで外部アクセス用URLを発行
print("外部アクセス用URLを準備中...")
subprocess.run(["npm", "install", "-g", "localtunnel"], stdout=subprocess.DEVNULL)
tunnel = subprocess.Popen(
    ["npx", "localtunnel", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# 生成されたURLを出力
for line in tunnel.stdout:
    if "your url is:" in line:
        url = line.split("your url is:")[1].strip()
        print("\n" + "=" * 50)
        print(f"★ 質疑応答API エンドポイントURL:\n{url}/api/chat")
        print("=" * 50)
        break

外部アクセス用URLを準備中...


INFO:     Started server process [998]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



★ 質疑応答API エンドポイントURL:
https://purple-streets-press.loca.lt/api/chat


In [ ]:
import requests

# 発行されたlocaltunnelのURLを指定
API_URL = "https://purple-streets-press.loca.lt/api/chat"

# 必須ヘッダー（tunnelの確認画面を自動回避）
headers = {
    "Content-Type": "application/json",
    "bypass-tunnel-reminder": "true",
}

# 送信する質問データ
payload = {"message": "学校のプログラミング授業で使えるアイデアを教えて！"}

# APIへ送信して回答を取得
response = requests.post(API_URL, json=payload, headers=headers)
data = response.json()

# AIからの回答を表示
if data.get("status") == "success":
    print("【AIの回答】")
    print(data["response"])
else:
    print("エラーが発生しました:", data.get("message"))

INFO:     34.50.163.26:0 - "POST /api/chat HTTP/1.1" 200 OK
【AIの回答】
ご指示に従って、学校用のプログラミング授業アイデアを調べました。以下に、対象学年や目的に合わせた授業アイデアとおすすめの教材をまとめてご提案します。

### 1. 学年別の授業アイデア

#### 小学校向け

*   **「順序」と「条件分岐」を学ぶ（算数×プログラミング）**
    *   **内容：** Scratchなどを使用し、正多角形（正三角形や正方形）を画面上のキャラクターに描かせる。
    *   **学び：** 「外角」の概念や、「繰り返す」というプログラミングの効率化を直感的に理解する。
*   **センサーを使った省エネ体験（理科×プログラミング）**
    *   **内容：** 小型コンピューター「micro:bit（マイクロビット）」を用い、「人が近づいたらライトがつく」「暗くなったらライトがつく」仕組みを作


In [ ]:
import requests

# 発行されたlocaltunnelのURLを指定
API_URL = "https://xxxx.loca.lt/api/chat"

# 必須ヘッダー（tunnelの確認画面を自動回避）
headers = {
    "Content-Type": "application/json",
    "bypass-tunnel-reminder": "true",
}

# 送信する質問データ
payload = {"message": "質問"}

# APIへ送信して回答を取得
response = requests.post(API_URL, json=payload, headers=headers)
data = response.json()

# AIからの回答を表示
if data.get("status") == "success":
    print("【AIの回答】")
    print(data["response"])
else:
    print("エラーが発生しました:", data.get("message"))